# CytoSignal diffusion scores only: 100, 200, and 400 µm

This notebook calculates only `diffusion-Raw_smooth` LR scores within each `EB_manual`. It does not run permutation or significance testing. Each radius is calculated independently, and cells from different EBs are never mixed.

In [20]:
options(width = 120)

library(cytosignal)
library(hdf5r)
library(Matrix)

cat("R:", R.version.string, "\n")
cat("cytosignal:", as.character(packageVersion("cytosignal")), "\n")

R: R version 4.3.3 (2024-02-29) 
cytosignal: 0.5.1 


In [21]:
H5AD_FILE <- "/Users/newuser/test/rna_for_cytosignal.h5ad"
OUTPUT_DIR <- "/Users/newuser/test/cytosignal_diffusion_scores_only"

SCALE_FACTOR <- 1
DIFFUSION_RADII_UM <- c(100, 200, 400)

stopifnot(file.exists(H5AD_FILE))
dir.create(OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
cat("Radii:", paste(DIFFUSION_RADII_UM, collapse = ", "), "µm\n")

Radii: 100, 200, 400 µm


## Load the H5AD and `EB_manual`

In [22]:
cs_full <- H5ADToCS(
  filename = H5AD_FILE,
  layer = "layers/counts",
  spatialKey = "X_spatial",
  clusterKey = "cell_type"
)

read_h5ad_categorical <- function(filename, key) {
  h5 <- H5File$new(filename, mode = "r")
  on.exit(h5$close_all())
  node <- h5[[paste0("obs/", key)]]
  if (!inherits(node, "H5Group") ||
      !all(c("categories", "codes") %in% names(node))) {
    stop("obs['", key, "'] is not stored as an H5AD categorical column")
  }
  categories <- as.character(node[["categories"]][])
  codes <- as.integer(node[["codes"]][])
  values <- rep(NA_character_, length(codes))
  valid <- codes >= 0L
  values[valid] <- categories[codes[valid] + 1L]
  values
}

eb_manual <- read_h5ad_categorical(H5AD_FILE, "EB_manual")
names(eb_manual) <- colnames(cs_full@raw.counts)

stopifnot(length(eb_manual) == ncol(cs_full@raw.counts))
stopifnot(identical(rownames(cs_full@cells.loc), colnames(cs_full@raw.counts)))

EB_IDS <- setdiff(unique(eb_manual), c("Unassigned", NA_character_))
EB_IDS <- EB_IDS[order(as.integer(sub("^EB_", "", EB_IDS)))]
cat("Manual EBs:", length(EB_IDS), "\n")

Manual EBs: 51 


## Scores-only functions

`inferScoreLR()` calculates the real score without a null distribution. The final matrix multiplication reproduces CytoSignal's deterministic smoothing of the real score, without calling its permutation-dependent null-score code.

In [23]:
prepare_eb_cs <- function(eb_id, counts_thresh = 300, gene_thresh = 50) {
  if (!eb_id %in% EB_IDS) stop("Unknown EB_manual value: ", eb_id)
  keep <- which(eb_manual == eb_id)
  if (length(keep) < 3L) stop(eb_id, " has fewer than three cells")

  object <- createCytoSignal(
    raw.data = cs_full@raw.counts[, keep, drop = FALSE],
    cells.loc = cs_full@cells.loc[keep, , drop = FALSE],
    clusters = droplevels(cs_full@clusters[keep])
  )

  upper_names <- toupper(rownames(object@raw.counts))
  if (anyDuplicated(upper_names)) {
    stop("Upper-casing gene names creates duplicates in ", eb_id)
  }
  rownames(object@raw.counts) <- upper_names

  object <- addIntrDB(object, g_to_u, db.diff, db.cont, inter.index)
  object <- removeLowQuality(
    object,
    counts.thresh = counts_thresh,
    gene.thresh = gene_thresh
  )
  object <- changeUniprot(object)
  object
}

findNNRaw_with_jitter <- function(object, eb_id, jitter_um = 1e-4) {
  original_loc <- object@cells.loc
  xy <- as.data.frame(original_loc[, 1:2, drop = FALSE])

  # Keep the first cell at each coordinate unchanged and move only later duplicates.
  duplicate_cells <- duplicated(xy)
  n_jittered <- sum(duplicate_cells)

  if (n_jittered > 0L) {
    # A fixed EB-specific seed gives identical jitter at all three radii.
    eb_number <- as.integer(sub("^EB_", "", eb_id))
    set.seed(20260000L + eb_number)
    angles <- runif(n_jittered, 0, 2 * pi)
    delta <- jitter_um * cbind(cos(angles), sin(angles))
    object@cells.loc[duplicate_cells, 1:2] <-
      original_loc[duplicate_cells, 1:2, drop = FALSE] + delta
    message(eb_id, ": applied ", jitter_um, " µm jitter to ",
            n_jittered, " duplicated-coordinate cell(s)")
  }

  object <- findNNRaw(object)
  # The original H5AD and reported coordinates remain unchanged. The neighbor
  # graphs retain the negligible jitter needed for Delaunay triangulation.
  object@cells.loc <- original_loc

  list(
    object = object,
    duplicate_coordinate_cells = n_jittered,
    dt_jitter_um = if (n_jittered > 0L) jitter_um else 0
  )
}

score_one_radius <- function(base_object, eb_id, radius_um) {
  object <- base_object

  object <- inferEpsParams(
    object,
    scale.factor = SCALE_FACTOR,
    r.eps.real = radius_um
  )

  # findNNRaw() requires the current radius parameters to exist first.
  nn_result <- findNNRaw_with_jitter(object, eb_id)
  object <- nn_result$object
  object <- findNNGauEB(object, imp.name = "diffusion")
  object <- imputeNiche(object, imp.use = "diffusion", weights = "none")

  object <- inferScoreLR(
    object,
    lig.imp = "diffusion",
    recep.imp = "Raw",
    norm.method = "default",
    intr.db.use = "diff_dep",
    lrscore.name = "diffusion-Raw"
  )

  score_raw <- object@lrscore[["diffusion-Raw"]]@score
  diffusion_graph <- object@imputation[["diffusion"]]@nn.graph

  # Exact real-score smoothing operation used by CytoSignal::smoothScoreLR().
  score_smooth <- Matrix::Matrix(
    t(t(score_raw) %*% diffusion_graph),
    sparse = TRUE
  )
  dimnames(score_smooth) <- dimnames(score_raw)

  result <- list(
    EB = eb_id,
    radius_um = radius_um,
    sigma_um = object@parameters$sigma.scale,
    score_slot = "diffusion-Raw_smooth",
    duplicate_coordinate_cells = nn_result$duplicate_coordinate_cells,
    dt_jitter_um = nn_result$dt_jitter_um,
    score = score_smooth
  )

  rds_file <- file.path(
    OUTPUT_DIR,
    sprintf("%s_diffusion-Raw_smooth_radius%03dum.rds", eb_id, radius_um)
  )
  saveRDS(result, rds_file)

  # The CSV contains nonzero entries only; the sparse RDS preserves the full matrix.
  triplet <- Matrix::summary(score_smooth)
  n_nonzero <- nrow(triplet)
  long <- data.frame(
    EB = rep(eb_id, n_nonzero),
    radius_um = rep(radius_um, n_nonzero),
    cell_id = rownames(score_smooth)[triplet$i],
    lr_type = rep("diffusion", n_nonzero),
    score_slot = rep("diffusion-Raw_smooth", n_nonzero),
    interaction_id = colnames(score_smooth)[triplet$j],
    lr_score = triplet$x,
    stringsAsFactors = FALSE
  )
  csv_file <- sub("\\.rds$", "_nonzero.csv.gz", rds_file)
  con <- gzfile(csv_file, open = "wt")
  write.csv(long, con, row.names = FALSE)
  close(con)

  message(
    eb_id, " | radius=", radius_um, " µm | sigma=",
    round(result$sigma_um, 2), " µm | matrix=",
    nrow(score_smooth), " x ", ncol(score_smooth)
  )
  message("Saved: ", rds_file)
  message("Saved: ", csv_file)
  result
}

expected_result_files <- function(eb_id, radius_um) {
  prefix <- sprintf("%s_diffusion-Raw_smooth_radius%03dum", eb_id, radius_um)
  file.path(
    OUTPUT_DIR,
    c(paste0(prefix, ".rds"), paste0(prefix, "_nonzero.csv.gz"))
  )
}

run_radius_scan <- function(eb_id, radii_um = DIFFUSION_RADII_UM) {
  radius_complete <- vapply(
    radii_um,
    function(radius_um) all(file.exists(expected_result_files(eb_id, radius_um))),
    logical(1)
  )
  missing_radii <- radii_um[!radius_complete]

  if (length(missing_radii) == 0L) {
    message("Skipping completed EB: ", eb_id)
    return(invisible(NULL))
  }

  message("Preparing ", eb_id, "; missing radii: ",
          paste(missing_radii, collapse = ", "), " µm")
  base_object <- prepare_eb_cs(eb_id)
  for (radius_um in missing_radii) {
    result <- score_one_radius(base_object, eb_id, radius_um)
    rm(result)
    gc()
  }
  invisible(NULL)
}

## Run one EB first

This runs all three radii for `EB_1`. It performs no permutation.

In [24]:
TARGET_EB <- "EB_1"
eb1_scores <- run_radius_scan(TARGET_EB)

data.frame(
  radius_um = vapply(eb1_scores, `[[`, numeric(1), "radius_um"),
  sigma_um = vapply(eb1_scores, `[[`, numeric(1), "sigma_um"),
  cells = vapply(eb1_scores, function(x) nrow(x$score), integer(1)),
  interactions = vapply(eb1_scores, function(x) ncol(x$score), integer(1))
)

Skipping completed EB: EB_1



radius_um,sigma_um,cells,interactions
<dbl>,<dbl>,<int>,<int>


## Resumable batch: complete only missing EB/radius results

A radius is complete only when both its `.rds` and `_nonzero.csv.gz` exist. Fully completed EBs are skipped, and a partially completed EB reruns only incomplete radii. For exact duplicate coordinates, only the later duplicate cell receives a reproducible `1e-4` µm jitter in memory for Delaunay construction; the input H5AD and reported coordinates are not modified.

In [25]:
RUN_ALL_EBS <- TRUE

if (RUN_ALL_EBS) {
  failures <- list()
  for (eb_id in EB_IDS) {
    tryCatch({
      result <- run_radius_scan(eb_id)
      rm(result)
      gc()
    }, error = function(e) {
      failures[[eb_id]] <<- conditionMessage(e)
      message("FAILED ", eb_id, ": ", conditionMessage(e))
    })
  }
  print(failures)
} else {
  message("Full batch not started.")
}

Skipping completed EB: EB_1

Skipping completed EB: EB_2

Skipping completed EB: EB_3

Skipping completed EB: EB_4

Skipping completed EB: EB_5

Skipping completed EB: EB_6

Skipping completed EB: EB_7

Skipping completed EB: EB_8

Skipping completed EB: EB_9

Skipping completed EB: EB_10

Skipping completed EB: EB_11

Skipping completed EB: EB_12

Skipping completed EB: EB_13

Skipping completed EB: EB_14

Skipping completed EB: EB_15

Skipping completed EB: EB_16

Skipping completed EB: EB_17

Skipping completed EB: EB_18

Skipping completed EB: EB_19

Skipping completed EB: EB_20

Skipping completed EB: EB_21

Skipping completed EB: EB_22

Skipping completed EB: EB_23

Skipping completed EB: EB_24

Skipping completed EB: EB_25

Skipping completed EB: EB_26

Skipping completed EB: EB_27

Skipping completed EB: EB_28

Skipping completed EB: EB_29

Skipping completed EB: EB_30

Skipping completed EB: EB_31

Skipping completed EB: EB_32

Skipping completed EB: EB_33

Skipping completed 

list()
